# Test the video → 3D box → speed pipeline on real I-24 video

Runs `scripts/video_speed_cli.py` on one I-24 camera video, then scores it against that scene's
annotations with `scripts/eval_video_cli.py`:

- **Calibration check**: which `--calib-units` / `--calib-image-scale` makes the annotated cuboids land on the
  YOLO detections. If it disagrees with the first run, the run is repeated with the right flags. Detections are
  reused, so YOLO doesn't run again.
- **Per-window speed error** vs. the annotation-derived speed (the training target), for:
  the model on video-lifted boxes, plain geometry on those boxes, and the model on the annotation boxes.

Use a GPU runtime (Runtime → Change runtime type → T4).

In [ ]:
# ---- settings ----
SCENE = 'scene1'
CAMERA = 'p1c1'
MAX_FRAMES = 1800          # ~1 minute at 30 fps; None = whole video
WEIGHTS = 'yolo11m.pt'     # yolo11n.pt is faster, m detects small/far vehicles better
BRANCH = 'video-3d-bbox'
MODE = '3d'                # which checkpoint to test: 2d / 3d / combined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT = None
for c in ['/content/drive/MyDrive/Final-Project-CHULA', '/content/drive/Shareddrives/Final-Project-CHULA']:
    if os.path.isdir(c):
        PROJECT = c
        break
if PROJECT is None:  # shared-with-me folders need a shortcut in MyDrive
    for root, dirs, _ in os.walk('/content/drive/MyDrive'):
        if 'Final-Project-CHULA' in dirs:
            PROJECT = os.path.join(root, 'Final-Project-CHULA')
            break
assert PROJECT, 'Add a shortcut to Final-Project-CHULA in MyDrive, or set PROJECT manually'

DATA_DIR, VIDEO = None, None
for root, dirs, files in os.walk(PROJECT):
    if DATA_DIR is None and {'obj', 'ts', 'hg'}.issubset(dirs):
        DATA_DIR = root
    if VIDEO is None and os.path.basename(root) == SCENE and f'{CAMERA}.mp4' in files:
        VIDEO = os.path.join(root, f'{CAMERA}.mp4')
print('PROJECT =', PROJECT)
print('DATA_DIR =', DATA_DIR)
print('VIDEO    =', VIDEO)
assert DATA_DIR and VIDEO

In [ ]:
%cd /content
!rm -rf capcapcar-speed && git clone -q -b {BRANCH} https://github.com/WalkerCze3/capcapcar-speed.git
%cd capcapcar-speed
!pip install -q -r requirements-video.txt

## Checkpoint

Uses `Final-Project-CHULA/runs/v2/<MODE>/best.pt` from Drive if it exists. Otherwise it trains one (~10 min on a
T4) and saves it to Drive so later runs skip this step.

In [ ]:
import shutil
CKPT_DRIVE = f'{PROJECT}/runs/v2/{MODE}/best.pt'
CKPT = f'runs/v2/{MODE}/best.pt'
if os.path.exists(CKPT_DRIVE):
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    shutil.copy(CKPT_DRIVE, CKPT)
    print('using', CKPT_DRIVE)
else:
    !python scripts/train_v2_cli.py --data-dir "{DATA_DIR}" --mode {MODE} --out runs/v2/{MODE} --epochs 10
    os.makedirs(os.path.dirname(CKPT_DRIVE), exist_ok=True)
    shutil.copy(CKPT, CKPT_DRIVE)
    print('saved', CKPT_DRIVE)

## Run: detect + track → 3D boxes → model

In [ ]:
import json, subprocess

RUN = f'runs/video/{SCENE}_{CAMERA}'
HG = f'{DATA_DIR}/hg/{SCENE}_hg.json'
TS = f'{DATA_DIR}/ts/{SCENE}_ts.csv'

def run_pipeline(units='ft', image_scale=1.0, reuse_detections=False):
    cmd = ['python', 'scripts/video_speed_cli.py', '--video', VIDEO, '--checkpoint', CKPT,
           '--calib', HG, '--camera', CAMERA, '--ts-csv', TS, '--weights', WEIGHTS,
           '--calib-units', units, '--calib-image-scale', str(image_scale), '--out-dir', RUN, '--render']
    if MAX_FRAMES:
        cmd += ['--max-frames', str(MAX_FRAMES)]
    if reuse_detections:
        shutil.copy(f'{RUN}/detections.csv', '/content/detections_cache.csv')
        cmd += ['--detections', '/content/detections_cache.csv']
    subprocess.run(cmd, check=True)

def run_eval():
    subprocess.run(['python', 'scripts/eval_video_cli.py', '--run-dir', RUN, '--data-dir', DATA_DIR,
                    '--scene', SCENE, '--camera', CAMERA], check=True)
    return json.load(open(f'{RUN}/eval_summary.json'))

run_pipeline()
summary = run_eval()

In [ ]:
# If the annotations line up better under other calibration settings, redo the lifting with them.
if summary['calibration_mismatch']:
    best = summary['calibration_best']
    print('re-running lifting with', best)
    run_pipeline(best['units'], best['image_scale'], reuse_detections=True)
    summary = run_eval()
if summary['calibration_best']['median_iou'] < 0.3:
    print('WARNING: annotations barely overlap detections under every calibration hypothesis --'
          ' check the video/annotation frame alignment (eval_video_cli.py --frame-offset).')

## Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print(pd.read_csv(f'{RUN}/calibration_check.csv').to_string(index=False), '\n')
table = pd.DataFrame({k: summary[k] for k in ['model_video_mps', 'geometric_video_mps', 'model_on_annotations_mps']}).T
print(f"matched {summary['tracks_matched']}/{summary['tracks']} tracks, "
      f"evaluated {summary['windows_evaluated']}/{summary['windows']} windows, "
      f"mean GT speed {summary['gt_mean_mps']:.2f} m/s")
table

In [ ]:
ev = pd.read_csv(f'{RUN}/window_eval.csv')
fig, ax = plt.subplots(figsize=(6, 6))
lim = [0, max(ev[['gt_mps', 'speed_mps', 'geometric_mps']].max()) * 1.05]
ax.plot(lim, lim, color='gray', lw=1)
ax.scatter(ev['gt_mps'], ev['speed_mps'], s=10, label='model (video 3D boxes)')
ax.scatter(ev['gt_mps'], ev['geometric_mps'], s=10, alpha=0.6, label='geometry only (video 3D boxes)')
ax.scatter(ev['gt_mps'], ev['model_on_annotations_mps'], s=10, alpha=0.6, label='model (annotation boxes)')
ax.set(xlim=lim, ylim=lim, xlabel='annotation speed (m/s)', ylabel='predicted (m/s)', title=f'{SCENE} {CAMERA}')
ax.legend()
plt.show()

In [ ]:
# A few frames of the annotated video (cuboids + speed labels).
import cv2
cap = cv2.VideoCapture(f'{RUN}/annotated.mp4')
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fig, axes = plt.subplots(3, 1, figsize=(14, 24))
for ax, f in zip(axes, [n // 4, n // 2, 3 * n // 4]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, img = cap.read()
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(f'frame {f}'); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Keep the outputs (CSV/JSON + annotated.mp4) on Drive.
dst = f'{PROJECT}/runs/video/{SCENE}_{CAMERA}'
shutil.copytree(RUN, dst, dirs_exist_ok=True)
print('copied to', dst)